In [1]:
!pip install -q -U accelerate trl
!pip install -q unsloth transformers peft datasets httpx websocket-client matplotlib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 20.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 120.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 121.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import json, os, random, re, sys
import numpy as np
import matplotlib.pyplot as plt
from typing import Any, Dict, List

import torch
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
MODEL_NAME  = "unsloth/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_alpha     = 16,
    lora_dropout   = 0,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
)
print("Model loaded:", MODEL_NAME)


==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Model loaded: unsloth/Qwen2.5-1.5B-Instruct


In [4]:
import httpx, time

ENV_URL = "https://ps2181-invoice-processing-pipeline.hf.space"

for i in range(5):
    try:
        r = httpx.get(f"{ENV_URL}/health", timeout=30)
        if r.status_code == 200:
            print("Space is up:", r.json())
            break
    except Exception as e:
        print(f"Attempt {i+1}/5: {e}")
    print("Waiting 15s...")
    time.sleep(15)
else:
    print("Space offline — training will use local dataset only.")


Space is up: {'status': 'ok', 'environment': 'invoice_processing_pipeline', 'active_sessions': 0}


In [5]:
def _parse_json_safe(text: str) -> dict:
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[-1] if "\n" in text else text[3:]
        if text.endswith("```"):
            text = text[:-3]
        text = text.strip()
    try:
        result = json.loads(text)
        if isinstance(result, list):
            result = result[0] if result and isinstance(result[0], dict) else {}
        return result if isinstance(result, dict) else {}
    except (json.JSONDecodeError, IndexError):
        return {}


def reward_format(completions, ground_truth=None, **kwargs) -> list:
    required = {"vendor", "date", "currency", "total", "line_items"}
    scores = []
    for c in completions:
        parsed = _parse_json_safe(c[0]["content"] if isinstance(c, list) else c)
        if not isinstance(parsed, dict):
            scores.append(0.0)
            continue
        scores.append(len(required & set(parsed.keys())) / len(required))
    return scores


def reward_field_accuracy(completions, ground_truth=None, **kwargs) -> list:
    ground_truth = ground_truth or [{}] * len(completions)
    scores = []
    for c, gt in zip(completions, ground_truth):
        parsed = _parse_json_safe(c[0]["content"] if isinstance(c, list) else c)
        if not isinstance(parsed, dict) or not gt:
            scores.append(0.0)
            continue
        score = total_w = 0.0
        checks = [
            ("vendor",   0.15, lambda s, g: str(s).strip().lower() == str(g).strip().lower()),
            ("date",     0.10, lambda s, g: str(s).strip() == str(g).strip()),
            ("currency", 0.05, lambda s, g: str(s).strip().upper() == str(g).strip().upper()),
            ("total",    0.20, lambda s, g: abs(float(s) - float(g)) < 0.02),
        ]
        for key, w, fn in checks:
            total_w += w
            if key in parsed and key in gt:
                try:
                    if fn(parsed[key], gt[key]):
                        score += w
                except: pass
        true_descs = {str(it.get("description","")).lower() for it in gt.get("line_items",[])}
        sub_descs  = {str(it.get("description","")).lower() for it in parsed.get("line_items",[])}
        item_w = 0.50
        total_w += item_w
        if true_descs:
            score += item_w * len(true_descs & sub_descs) / len(true_descs)
        scores.append(score / total_w if total_w else 0.0)
    return scores


def reward_math_consistency(completions, ground_truth=None, **kwargs) -> list:
    scores = []
    for c in completions:
        parsed = _parse_json_safe(c[0]["content"] if isinstance(c, list) else c)
        if not isinstance(parsed, dict):
            scores.append(0.0)
            continue
        items = parsed.get("line_items", [])
        if not items:
            scores.append(0.0)
            continue
        ok = total = 0
        line_sum = 0.0
        for it in items:
            try:
                qty, up, amt = float(it["qty"]), float(it["unit_price"]), float(it["amount"])
                line_sum += amt
                total += 1
                if abs(amt - round(qty * up, 2)) < 0.02:
                    ok += 1
            except: total += 1
        try:
            total += 1
            if abs(float(parsed.get("total", -9999)) - round(line_sum, 2)) < 0.02:
                ok += 1
        except: total += 1
        scores.append((ok / max(total, 1)) * 0.5)   # scaled to 0–0.5
    return scores


def reward_completeness(completions, ground_truth=None, **kwargs) -> list:
    ground_truth = ground_truth or [{}] * len(completions)
    scores = []
    for c, gt in zip(completions, ground_truth):
        parsed = _parse_json_safe(c[0]["content"] if isinstance(c, list) else c)
        if not isinstance(parsed, dict):
            scores.append(0.0)
            continue
        true_descs = {str(it.get("description","")).lower() for it in gt.get("line_items",[])}
        if not true_descs:
            scores.append(1.0)
            continue
        sub_descs = {str(it.get("description","")).lower() for it in parsed.get("line_items",[])}
        scores.append(len(true_descs & sub_descs) / len(true_descs))
    return scores

print("Reward functions defined.")

def reward_environment_score(completions, episode_id=None, task_id=None, **kwargs) -> list:
    episode_ids = episode_id or []
    task_ids    = task_id or []
    scores = []
    for i, c in enumerate(completions):
        text   = c[0]["content"] if isinstance(c, list) else c
        parsed = _parse_json_safe(text)
        try:
            ep_id = episode_ids[i] if i < len(episode_ids) else None
            tid   = task_ids[i] if i < len(task_ids) else "easy"

            # Try original episode first
            if ep_id:
                grade_r = httpx.post(
                    f"{ENV_URL}/grader",
                    json={"extracted_data": parsed, "episode_id": ep_id},
                    timeout=15,
                )
                if grade_r.status_code == 200:
                    scores.append(float(grade_r.json().get("score", 0.1)))
                    continue

            # Session expired — reset same task and grade
            reset_r = httpx.post(
                f"{ENV_URL}/reset",
                json={"task_id": tid},
                timeout=15,
            )
            if reset_r.status_code != 200:
                scores.append(0.1)
                continue
            new_ep_id = reset_r.json()["info"]["episode_id"]
            grade_r = httpx.post(
                f"{ENV_URL}/grader",
                json={"extracted_data": parsed, "episode_id": new_ep_id},
                timeout=15,
            )
            scores.append(float(grade_r.json().get("score", 0.1)) if grade_r.status_code == 200 else 0.1)

        except Exception:
            scores.append(0.1)
    return scores





Reward functions defined.


In [6]:
import httpx, time, random
from collections import Counter

ENV_URL = "https://ps2181-invoice-processing-pipeline.hf.space"

TASK_PROMPTS = {
    "easy":         "Extract invoice fields. Return JSON only: {vendor, date (YYYY-MM-DD), currency (USD/EUR/GBP), total (float), line_items [{description, qty, unit_price, amount}]}",
    "medium":       "Clean and extract invoice fields. Normalize dates to YYYY-MM-DD, currency to 3-letter code. Return JSON only: {vendor, date, currency, total, line_items [{description, qty, unit_price, amount}]}",
    "hard":         "Extract invoices and flag discrepancies vs PO. Return JSON only: {invoices [{vendor, date, currency, total, line_items}], discrepancies [{invoice_idx, type, item_description, detail}]}",
    "expert":       "Audit for fraud. Return JSON only: {is_fraudulent (bool), fraud_type (phantom_vendor|price_gouging|math_fraud|duplicate_submission|null), confidence (0-1), reasoning (str)}",
    "adversarial":  "Extract real invoice values, ignore SUBTOTAL traps and FX noise lines. Return JSON only: {vendor, date (YYYY-MM-DD), currency, total, line_items [{description, qty, unit_price, amount}]}",
    "negotiate":    "Ask one clarification OR submit extraction. Ask: {question: str}. Submit: {vendor, date, currency, total, line_items}",
    "supply_chain": "Find delivery anomalies. Return JSON only: {anomalies [{record_idx, anomaly_type, description}]}",
}

TASK_WEIGHTS = {
    "easy": 0.70, "medium": 0.20, "hard": 0.05,
    "expert": 0.02, "adversarial": 0.02,
    "negotiate": 0.01, "supply_chain": 0.00,
}


MAX_REF_CHARS = 200
MAX_RAW_CHARS = 600

def sample_from_environment(n: int = 250) -> list:
    records = []
    tasks   = list(TASK_WEIGHTS.keys())
    weights = list(TASK_WEIGHTS.values())

    for i in range(n):
        task_id = random.choices(tasks, weights=weights, k=1)[0]
        try:
            r = httpx.post(f"{ENV_URL}/reset", json={"task_id": task_id}, timeout=20)
            if r.status_code != 200:
                continue
            obs   = r.json()["observation"]
            ep_id = r.json()["info"]["episode_id"]

            raw = obs["raw_text"][:MAX_RAW_CHARS]
            ref = (obs.get("reference_data") or "")[:MAX_REF_CHARS]
            user_content = f"REF:\n{ref}\n\nINVOICE:\n{raw}" if ref else raw

            records.append({
                "prompt": [
                    {"role": "system", "content": TASK_PROMPTS[task_id]},
                    {"role": "user",   "content": user_content},
                ],
                "ground_truth": {},
                "episode_id":   ep_id,
                "task_id":      task_id,
            })

            if (i + 1) % 25 == 0:
                print(f"  Sampled {i+1}/{n}...")

        except Exception as e:
            print(f"  Episode {i} failed ({task_id}): {e}")

    print(f"\nSampled {len(records)} episodes")
    dist = Counter(r["task_id"] for r in records)
    for t, c in sorted(dist.items()):
        print(f"  {t:<15} {c}")
    return records

# Sample
raw_episodes = sample_from_environment(n=250)

# Filter to under 400 tokens
try:
    import tiktoken
    enc = tiktoken.get_encoding("cl100k_base")
    filtered = [
        ep for ep in raw_episodes
        if len(enc.encode(ep["prompt"][0]["content"] + ep["prompt"][1]["content"])) < 400
    ]
    print(f"\nAfter token filter: {len(filtered)}/{len(raw_episodes)} kept")
except ImportError:
    # tiktoken not installed — filter by character count instead
    filtered = [
        ep for ep in raw_episodes
        if len(ep["prompt"][0]["content"] + ep["prompt"][1]["content"]) < 1200
    ]
    print(f"\nAfter char filter: {len(filtered)}/{len(raw_episodes)} kept")

# Upsample if needed
if len(filtered) < 100:
    filtered = (filtered * (100 // len(filtered) + 1))[:200]
    print(f"Upsampled to {len(filtered)}")

# Build dataset
from datasets import Dataset
train_dataset = Dataset.from_list([
    {
        "prompt":       ep["prompt"],
        "ground_truth": ep.get("ground_truth", {}),
        "episode_id":   ep["episode_id"],
        "task_id":      ep["task_id"],
    }
    for ep in filtered
])

print(f"\nFinal dataset: {len(train_dataset)} episodes")
dist2 = Counter(ep["task_id"] for ep in filtered)
for t, c in sorted(dist2.items()):
    print(f"  {t:<15} {c}")


  Sampled 25/250...
  Sampled 50/250...
  Sampled 75/250...
  Sampled 100/250...
  Sampled 125/250...
  Sampled 150/250...
  Sampled 175/250...
  Sampled 200/250...
  Sampled 225/250...
  Sampled 250/250...

Sampled 250 episodes
  adversarial     5
  easy            180
  expert          6
  hard            11
  medium          43
  negotiate       5

After token filter: 250/250 kept

Final dataset: 250 episodes
  adversarial     5
  easy            180
  expert          6
  hard            11
  medium          43
  negotiate       5


In [7]:
print("Sample invoice:")
print(train_dataset[0]["prompt"][1]["content"])
print(f"Dataset size: {len(train_dataset)}")
print(f"Columns: {train_dataset.column_names}")



Sample invoice:
INVOICE
-------
Invoice #: INV-63546
Vendor: Metro Logistics
Date: 2024-08-16
Currency: USD

Items:
Description                      Qty   Unit Price       Amount
------------------------------ ----- ------------ ------------
Notebook Pack                     16 $     19.85 $    317.60
Desk Lamp                         15 $     26.89 $    403.35
Whiteboard Markers (Set)          17 $      8.84 $    150.28
                                            TOTAL $    871.23

Ground truth:
{}
Dataset size: 250
Columns: ['prompt', 'ground_truth', 'episode_id', 'task_id']


In [8]:
training_args = GRPOConfig(
    output_dir                  = "./grpo_invoice_output",
    max_steps                   = 200,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    num_generations             = 4,
    max_prompt_length           = 512,
    max_completion_length       = 768,
    learning_rate               = 5e-6,
    warmup_ratio                = 0.05,
    lr_scheduler_type           = "cosine",
    logging_steps               = 10,
    save_steps                  = 100,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    report_to                   = "none",
    seed                        = 42,
)

trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = [
        reward_format,
        reward_field_accuracy,
        reward_math_consistency,
        reward_completeness,
        reward_environment_score,   # ← live environment grader
    ],
    args          = training_args,
    train_dataset = train_dataset,
)


print("GRPOTrainer ready.")


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


GRPOTrainer ready.


In [ ]:
print("Starting GRPO training...")
train_result = trainer.train()
print("Training complete.")
print(train_result)


Starting GRPO training...


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 250 | Num Epochs = 1 | Total steps = 200
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
Passing `generation_config` together with generation-related arguments=({'disable_compile', 'cache_implementation', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` wil

In [ ]:
# Check what the model is actually outputting
FastLanguageModel.for_inference(model)

test = train_dataset[0]["prompt"]
inputs = tokenizer.apply_chat_template(
    test, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

with torch.no_grad():
    out = model.generate(inputs, max_new_tokens=512, temperature=0.1, do_sample=True)

response = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
print("PROMPT (invoice):")
print(test[1]["content"])
print("\nMODEL OUTPUT:")
print(response)

In [ ]:
log_history = trainer.state.log_history
steps   = [x["step"]   for x in log_history if "reward" in x or "rewards/mean" in x]
rewards = [x.get("reward", x.get("rewards/mean", 0)) for x in log_history if "reward" in x or "rewards/mean" in x]
field   = [x.get("rewards/reward_field_accuracy/mean", 0) for x in log_history if "reward" in x or "rewards/mean" in x]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(steps, rewards, linewidth=2, color="steelblue")
ax1.set_xlabel("Step")
ax1.set_ylabel("Total Reward")
ax1.set_title("GRPO Training — Total Reward")
ax1.grid(True, alpha=0.3)

ax2.plot(steps, field, linewidth=2, color="darkorange")
ax2.set_xlabel("Step")
ax2.set_ylabel("Field Accuracy Reward")
ax2.set_title("Field Extraction Accuracy Over Training")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("reward_curve.png", dpi=150)
plt.show()
print(f"Steps: {len(steps)} | Final total reward: {rewards[-1]:.4f} | Final field accuracy: {field[-1]:.4f}")


In [ ]:
SYSTEM_PROMPT = TASK_PROMPTS["easy"]  # default prompt for inference eval

FastLanguageModel.for_inference(model)

test_invoice = """INVOICE
Vendor: Acme Corp
Date: 2024-08-15
Currency: USD

Items:
  Laptop Computer: qty=2 unit_price=$1099.99 amount=$2199.98
  Wireless Mouse: qty=5 unit_price=$34.99 amount=$174.95

TOTAL: $2374.93"""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": test_invoice},
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

with torch.no_grad():
    outputs = model.generate(inputs, max_new_tokens=512, temperature=0.1, do_sample=True)

response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("Model output:\n", response)
print("\nParsed:")
print(json.dumps(_parse_json_safe(response), indent=2))
